# Citi Bike NYC — Dimensión: Duración de los Viajes (U1 Batch)
**Responsable:** Jose Miguel Condo Huamani  
**Arquitectura:** Medallion (Bronze → Silver → Gold) sobre Apache Spark (PySpark)  
**Metodología:** CRISP-DM | **Alcance:** Procesamiento Batch (U1)

---

## 1. Pregunta Central de Negocio
> ¿Cómo varía históricamente la duración de los viajes de Citi Bike y qué duración puede esperarse según las características del trayecto, temporalidad y perfil del usuario?

## 2. Variable Objetivo
* **`duration_minutes`**: Variable continua calculada como `(ended_at - started_at) / 60.0`.

## 3. Decisiones Operativas Habilitadas
1. **Balanceo de Flota:** Estimar tiempos de liberación de bicicletas en estaciones críticas para optimizar cuadrillas de rebalanceo.
2. **Gestión de Riesgos Tarifarios:** Cuantificar la probabilidad y volumen de viajes que exceden el umbral gratuito (30 min casual / 45 min member).
3. **Mantenimiento y Auditoría:** Distinguir averías mecánicas tempranas (< 1 min) frente a retenciones o extravíos anómalos (> 180 min).

In [1]:
from pyspark.sql import SparkSession

# 1. Configuración de SparkSession optimizada para procesamiento local/clúster
spark = SparkSession.builder \
    .appName("CitiBike-Batch-Duration-JoseCondo") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.default.parallelism", "16") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✅ SparkSession inicializada y configurada correctamente.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/11 16:07:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ SparkSession inicializada y configurada correctamente.


## Fase 2: Data Understanding — Ingesta a Capa Bronze
Lectura de los datos crudos con esquema explícito obligatorio para evitar el costo de inferSchema sobre millones de registros.

In [2]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

bronze_schema = StructType([
    StructField("ride_id", StringType(), False),
    StructField("rideable_type", StringType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("ended_at", TimestampType(), True),
    StructField("start_station_name", StringType(), True),
    StructField("start_station_id", StringType(), True),
    StructField("end_station_name", StringType(), True),
    StructField("end_station_id", StringType(), True),
    StructField("start_lat", DoubleType(), True),
    StructField("start_lng", DoubleType(), True),
    StructField("end_lat", DoubleType(), True),
    StructField("end_lng", DoubleType(), True),
    StructField("member_casual", StringType(), True)
])

csv_source_path = "/opt/data/citibike"
bronze_parquet_path = "/opt/dimencion/citibike_raw_parquet"

df_bronze_raw = spark.read.format("csv") \
    .option("header", "true") \
    .schema(bronze_schema) \
    .load(csv_source_path)

# Persistencia inmutable
df_bronze_raw.write.mode("ignore").parquet(bronze_parquet_path)

df_bronze = spark.read.parquet(bronze_parquet_path)
total_bronze = df_bronze.count()
print(f"✅ Capa Bronze cargada. Total filas crudas: {total_bronze:,}")
df_bronze.show(5, truncate=False)

26/09/11 16:07:25 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /opt/data/citibike.
java.io.FileNotFoundException: File /opt/data/citibike does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.analysis.ResolveDa

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/opt/data/citibike. SQLSTATE: 42K03

In [ ]:
from pyspark.sql.functions import (
    col, unix_timestamp, hour, dayofweek, month, to_date,
    sqrt, pow, round as spark_round, when, approx_count_distinct
)

# Derivación de features temporales, de demanda y espaciales
df_bronze_features = df_bronze \
    .withColumn("duration_minutes", spark_round((unix_timestamp(col("ended_at")) - unix_timestamp(col("started_at"))) / 60.0, 2)) \
    .withColumn("start_date", to_date(col("started_at"))) \
    .withColumn("start_hour", hour(col("started_at"))) \
    .withColumn("day_of_week", dayofweek(col("started_at"))) \
    .withColumn("start_month", month(col("started_at"))) \
    .withColumn("is_weekend", when(col("day_of_week").isin(1, 7), 1).otherwise(0)) \
    .withColumn("is_rush_hour", when(col("start_hour").isin(8, 9, 17, 18), 1).otherwise(0)) \
    .withColumn("time_window", when(col("start_hour").between(6, 11), "morning")
                               .when(col("start_hour").between(12, 17), "afternoon")
                               .when(col("start_hour").between(18, 22), "evening")
                               .otherwise("night")) \
    .withColumn("approx_distance_km", spark_round(
        sqrt(pow((col("end_lat") - col("start_lat")) * 111.0, 2) + 
             pow((col("end_lng") - col("start_lng")) * 85.0, 2)), 3)
    )

print("=== CARDINALIDAD EN BRONZE ===")
df_bronze_features.select(
    approx_count_distinct("ride_id").alias("viajes_unicos"),
    approx_count_distinct("member_casual").alias("tipos_usuario"),
    approx_count_distinct("rideable_type").alias("tipos_bici"),
    approx_count_distinct("start_station_id").alias("estaciones_origen")
).show()

In [ ]:
from pyspark.sql.functions import expr, skewness, kurtosis, count

print("=== DISTRIBUCIÓN ESTADÍSTICA ROBUSTA (DURACIÓN) ===")

# Percentiles, IQR, Asimetría y Curtosis
df_distribucion = df_bronze_features.filter(col("duration_minutes").isNotNull()).select(
    spark_round(expr("percentile_approx(duration_minutes, 0.25)"), 2).alias("p25"),
    spark_round(expr("percentile_approx(duration_minutes, 0.50)"), 2).alias("p50_mediana"),
    spark_round(expr("percentile_approx(duration_minutes, 0.75)"), 2).alias("p75"),
    spark_round(expr("percentile_approx(duration_minutes, 0.90)"), 2).alias("p90"),
    spark_round(expr("percentile_approx(duration_minutes, 0.99)"), 2).alias("p99"),
    spark_round(skewness("duration_minutes"), 4).alias("skewness"),
    spark_round(kurtosis("duration_minutes"), 4).alias("kurtosis")
).withColumn("iqr", spark_round(col("p75") - col("p25"), 2))

df_distribucion.show()

print("=== HISTOGRAMA POR INTERVALOS DE TIEMPO ===")
df_bronze_features.withColumn("duration_bin", 
    when(col("duration_minutes") < 1.0, "0. < 1 min (Averia)")
    .when(col("duration_minutes").between(1.0, 5.0), "1. [1-5 min)")
    .when(col("duration_minutes").between(5.0, 15.0), "2. [5-15 min)")
    .when(col("duration_minutes").between(15.0, 30.0), "3. [15-30 min)")
    .when(col("duration_minutes").between(30.0, 45.0), "4. [30-45 min)")
    .when(col("duration_minutes").between(45.0, 180.0), "5. [45-180 min)")
    .otherwise("6. > 180 min (Retencion)")
).groupBy("duration_bin").agg(
    count("ride_id").alias("total_viajes"),
    spark_round(count("ride_id") * 100.0 / total_bronze, 2).alias("pct_del_total")
).orderBy("duration_bin").show(truncate=False)

## Fase 3: Data Preparation — Capa Silver (`silver.trips_duration_clean`)

### Reglas de Calidad y Saneamiento:
1. **Rango Operativo Válido:** Retención de viajes entre `[1.0, 180.0]` minutos. Registros fuera de rango se preservan en Bronze para alimentar la auditoría de fallas y retenciones.
2. **Integridad Geoespacial:** Filtro de coordenadas no nulas y desplazamiento mayor a cero (`approx_distance_km > 0.0`).
3. **Deduplicación Determinista:** Sobre `ride_id` conservando la última traza cronológica.
4. **Particionado Compuesto:** Por `member_casual` y `rideable_type` para prevenir asimetría en disco (*skew*) y maximizar el *partition pruning* sin sobrecargar la memoria.

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

print("=== AUDITORÍA PREVIA DE CALIDAD EN BRONZE ===")
df_auditoria = df_bronze_features.select(
    count(when(col("duration_minutes").isNull(), 1)).alias("nulos_duracion"),
    count(when(col("duration_minutes") < 1.0, 1)).alias("averias_menos_1min"),
    count(when(col("duration_minutes") > 180.0, 1)).alias("retenciones_mas_3h"),
    count(when(col("start_lat").isNull() | col("end_lat").isNull(), 1)).alias("nulos_gps"),
    count(when(col("approx_distance_km") <= 0.0, 1)).alias("distancia_cero")
)
df_auditoria.show()

# Saneamiento estricto
df_silver_clean = df_bronze_features.filter(
    col("ride_id").isNotNull() &
    col("duration_minutes").between(1.0, 180.0) &
    col("start_lat").isNotNull() & col("end_lat").isNotNull() &
    (col("approx_distance_km") > 0.0)
)

# Deduplicación por ride_id
window_ride = Window.partitionBy("ride_id").orderBy(col("started_at").desc())
df_silver_dedup = df_silver_clean \
    .withColumn("rn", row_number().over(window_ride)) \
    .filter(col("rn") == 1) \
    .drop("rn")

filas_silver = df_silver_dedup.count()
descartadas = total_bronze - filas_silver
print(f"Filas Bronze: {total_bronze:,}")
print(f"Filas Silver retenidas: {filas_silver:,} ({filas_silver / total_bronze * 100:.2f}%)")
print(f"Registros anómalos eliminados: {descartadas:,} ({descartadas / total_bronze * 100:.2f}%)")

In [ ]:
silver_parquet_dir = "/opt/dimencion/citibike_silver_parquet"

# Escritura particionada por member_casual y rideable_type
df_silver_dedup.write \
    .mode("overwrite") \
    .partitionBy("member_casual", "rideable_type") \
    .parquet(silver_parquet_dir)

print(f"✅ Capa Silver guardada exitosamente en: {silver_parquet_dir}")

df_silver = spark.read.parquet(silver_parquet_dir)
print("=== PLAN FÍSICO Y VERIFICACIÓN DE PARTITION PRUNING ===")
df_silver.filter((col("member_casual") == "member") & (col("rideable_type") == "electric_bike")).explain()

In [ ]:
from pyspark.sql.functions import avg

print("=== 📊 SEGMENTACIÓN DESCRIPTIVA (EXPLORACIÓN EN SILVER) ===")

print("--- 1. Duración media por DÍA DE LA SEMANA ---")
df_silver.groupBy("day_of_week").agg(
    spark_round(avg("duration_minutes"), 2).alias("duracion_media_min"),
    count("ride_id").alias("total_viajes")
).orderBy("day_of_week").show()

print("--- 2. Duración media por FRANJA HORARIA ---")
df_silver.groupBy("time_window").agg(
    spark_round(avg("duration_minutes"), 2).alias("duracion_media_min"),
    count("ride_id").alias("total_viajes")
).orderBy("time_window").show()

print("--- 3. Duración media: USUARIO × FIN DE SEMANA ---")
df_silver.withColumn("tipo_dia", when(col("is_weekend") == 1, "Fin de Semana").otherwise("Laboral")) \
    .groupBy("member_casual", "tipo_dia").agg(
        spark_round(avg("duration_minutes"), 2).alias("duracion_media_min"),
        count("ride_id").alias("total_viajes")
    ).orderBy("member_casual", "tipo_dia").show()

## Fase 4: Modeling & Fase 5: Evaluation — Pipeline ML sin Data Leakage

### Rigor Metodológico:
* **Split Train/Test:** La partición 80/20 se realiza **antes** de ajustar cualquier `StringIndexer` u `OneHotEncoder`.
* **Vector de Features Completo:** Incorpora `approx_distance_km`, `start_hour`, `day_of_week`, `start_month`, `is_weekend`, `is_rush_hour`, `time_window_vec`, `rideable_vec` y `member_vec`.
* **Benchmark Objetivo:** Se compara simultáneamente contra un Baseline Trivial (media observada en Train), Linear Regression (L2) y Random Forest Regressor.
* **Métricas Evaluadas:** RMSE, MAE, R² y MAPE.

In [ ]:
df_model_input = df_silver.select(
    "duration_minutes",
    "approx_distance_km",
    "start_hour",
    "day_of_week",
    "start_month",
    "is_weekend",
    "is_rush_hour",
    "time_window",
    "rideable_type",
    "member_casual"
).dropna()

# Muestra representativa de 10% para ajuste en entorno local/contenedor
df_sample_ml = df_model_input.sample(withReplacement=False, fraction=0.10, seed=42)

# Split Train/Test sin leakage
train_raw, test_raw = df_sample_ml.randomSplit([0.8, 0.2], seed=42)
print(f"Muestras de modelado: Train = {train_raw.count():,} | Test = {test_raw.count():,}")

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# 1. Codificadores categóricos
idx_rideable = StringIndexer(inputCol="rideable_type", outputCol="rideable_idx", handleInvalid="keep")
idx_member = StringIndexer(inputCol="member_casual", outputCol="member_idx", handleInvalid="keep")
idx_window = StringIndexer(inputCol="time_window", outputCol="time_window_idx", handleInvalid="keep")

encoder = OneHotEncoder(
    inputCols=["rideable_idx", "member_idx", "time_window_idx"],
    outputCols=["rideable_vec", "member_vec", "time_window_vec"],
    handleInvalid="keep"
)

# 2. VectorAssembler con features completas
assembler = VectorAssembler(
    inputCols=[
        "approx_distance_km",
        "start_hour",
        "day_of_week",
        "start_month",
        "is_weekend",
        "is_rush_hour",
        "rideable_vec",
        "member_vec",
        "time_window_vec"
    ],
    outputCol="features"
)

# 3. Ajuste exclusivo con train
prep_pipeline = Pipeline(stages=[idx_rideable, idx_member, idx_window, encoder, assembler])
prep_pipeline_model = prep_pipeline.fit(train_raw)

train_df = prep_pipeline_model.transform(train_raw)
test_df = prep_pipeline_model.transform(test_raw)
print("✅ Preprocesamiento vectorizado ajustado exclusivamente con Train.")

In [ ]:
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.sql.functions import avg, lit

# 1. Baseline trivial (Media fija de train)
mean_train_val = train_raw.select(avg("duration_minutes")).first()[0]
preds_baseline = test_df.withColumn("prediction", lit(mean_train_val))

# 2. Linear Regression (L2 / Ridge)
lr = LinearRegression(featuresCol="features", labelCol="duration_minutes", regParam=0.05, maxIter=30)
model_lr = lr.fit(train_df)
preds_lr = model_lr.transform(test_df)

# 3. Random Forest Regressor
rf = RandomForestRegressor(featuresCol="features", labelCol="duration_minutes", numTrees=30, maxDepth=8, seed=42)
model_rf = rf.fit(train_df)
preds_rf = model_rf.transform(test_df)
print("✅ Modelos entrenados y predicciones generadas sobre Test.")

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import abs as spark_abs

eval_rmse = RegressionEvaluator(labelCol="duration_minutes", predictionCol="prediction", metricName="rmse")
eval_mae = RegressionEvaluator(labelCol="duration_minutes", predictionCol="prediction", metricName="mae")
eval_r2 = RegressionEvaluator(labelCol="duration_minutes", predictionCol="prediction", metricName="r2")

def compute_metrics(name, df_preds):
    rmse = eval_rmse.evaluate(df_preds)
    mae = eval_mae.evaluate(df_preds)
    r2 = eval_r2.evaluate(df_preds)
    mape = df_preds.select(
        avg(spark_abs((col("duration_minutes") - col("prediction")) / col("duration_minutes")) * 100)
    ).first()[0]
    return (name, round(rmse, 3), round(mae, 3), round(r2, 3), round(mape, 2))

metricas = [
    compute_metrics("Baseline (Media Train)", preds_baseline),
    compute_metrics("Linear Regression (L2)", preds_lr),
    compute_metrics("Random Forest Regressor", preds_rf)
]

df_eval = spark.createDataFrame(metricas, ["Modelo", "RMSE (min)", "MAE (min)", "R2 Score", "MAPE (%)"])
print("=== COMPARATIVA FRENTE AL BASELINE (HITO S4) ===")
df_eval.show(truncate=False)

if metricas[2][1] < metricas[1][1]:
    best_model_name = "Random Forest Regressor"
    best_model = model_rf
    best_preds = preds_rf
else:
    best_model_name = "Linear Regression (L2)"
    best_model = model_lr
    best_preds = preds_lr

print(f">> Ganador por menor RMSE: {best_model_name}")

In [ ]:
print("=== ANÁLISIS DE RESIDUALES Y HETEROCEDASTICIDAD ===")

df_residuals = best_preds.select(
    "duration_minutes",
    "prediction",
    (col("duration_minutes") - col("prediction")).alias("residual")
).withColumn("abs_error", spark_abs(col("residual")))

# Estadísticas de distribución de residuales
df_residuals.select(
    spark_round(avg("residual"), 4).alias("media_residuales_sesgo"),
    spark_round(expr("stddev(residual)"), 4).alias("desv_residuales"),
    spark_round(expr("percentile_approx(residual, 0.50)"), 4).alias("mediana_residual"),
    spark_round(expr("percentile_approx(residual, 0.05)"), 4).alias("p05_residual"),
    spark_round(expr("percentile_approx(residual, 0.95)"), 4).alias("p95_residual")
).show()

# Diagnóstico de heterocedasticidad por tramos de duración
print("=== MAE SEGÚN RANGO DE DURACIÓN REAL ===")
df_residuals.withColumn("tramo_duracion",
    when(col("duration_minutes") < 10, "1. < 10 min")
    .when(col("duration_minutes").between(10, 30), "2. 10-30 min")
    .when(col("duration_minutes").between(30, 60), "3. 30-60 min")
    .otherwise("4. > 60 min")
).groupBy("tramo_duracion").agg(
    spark_round(avg("abs_error"), 2).alias("mae_tramo_min"),
    count("residual").alias("total_muestras")
).orderBy("tramo_duracion").show()

In [ ]:
# Exportación del PipelineModel completo para despliegue en Streamlit
full_prod_pipeline = Pipeline(stages=[idx_rideable, idx_member, idx_window, encoder, assembler, best_model])
full_prod_model = full_prod_pipeline.fit(train_raw)

model_export_dir = "/opt/dimencion/best_duration_model"
full_prod_model.write().overwrite().save(model_export_dir)
print(f"✅ Pipeline completo exportado exitosamente en: {model_export_dir}")

## Capa Gold: Data Marts para Decisiones Operativas y Streamlit

Se generan 4 Data Marts especializados y optimizados para responder a cada decisión operativa:
1. **`gold.fleet_turnover`:** Tiempos de liberación, mediana y percentil 75 por estación y franja.
2. **`gold.tariff_risks`:** Cuantificación del riesgo tarifario (> 30 min casual / > 45 min member).
3. **`gold.station_anomalies`:** Conteo de averías mecánicas tempranas (< 1 min) y retenciones anómalas (> 180 min) por estación origen (calculado desde Bronze).
4. **`gold.daily_timeseries`:** Serie temporal agregada por fecha con media móvil a 7 días.

In [ ]:
from pyspark.sql.functions import sum as spark_sum

print("=== GENERANDO DATA MARTS DE CAPA GOLD ===")

# 1. Data Mart: Balanceo de Flota y Tiempos de Liberación
df_gold_turnover = df_silver.groupBy(
    "start_station_id", "start_station_name", "time_window", "is_weekend"
).agg(
    spark_round(avg("duration_minutes"), 2).alias("duracion_media_min"),
    spark_round(expr("percentile_approx(duration_minutes, 0.50)"), 2).alias("mediana_duracion_min"),
    spark_round(expr("percentile_approx(duration_minutes, 0.75)"), 2).alias("p75_duracion_min"),
    spark_round(spark_sum("duration_minutes"), 2).alias("minutos_flota_en_uso"),
    count("ride_id").alias("total_despachos")
).filter(col("total_despachos") >= 50)

df_gold_turnover.write.mode("overwrite").parquet("/opt/dimencion/gold_fleet_turnover")
print(" 1. Guardado: /opt/dimencion/gold_fleet_turnover")

# 2. Data Mart: Riesgos Tarifarios
df_gold_tariffs = df_silver.groupBy("member_casual", "time_window", "day_of_week").agg(
    count("ride_id").alias("total_viajes"),
    spark_round(avg(when(
        (col("member_casual") == "casual") & (col("duration_minutes") > 30), 1)
        .when((col("member_casual") == "member") & (col("duration_minutes") > 45), 1)
        .otherwise(0)) * 100, 2).alias("pct_viajes_con_recargo"),
    spark_round(avg(when(
        (col("member_casual") == "casual") & (col("duration_minutes") > 30), col("duration_minutes") - 30)
        .when((col("member_casual") == "member") & (col("duration_minutes") > 45), col("duration_minutes") - 45)
        .otherwise(0)), 2).alias("exceso_promedio_min")
)

df_gold_tariffs.write.mode("overwrite").parquet("/opt/dimencion/gold_tariff_risks")
print(" 2. Guardado: /opt/dimencion/gold_tariff_risks")

# 3. Data Mart: Averías vs Retenciones (Calculado sobre Bronze para capturar anomalías reales)
df_gold_anomalies = df_bronze_features.groupBy("start_station_id", "start_station_name").agg(
    count(when(col("duration_minutes") < 1.0, 1)).alias("averias_menos_1min"),
    count(when(col("duration_minutes") > 180.0, 1)).alias("retenciones_mas_3h"),
    count("ride_id").alias("total_salidas")
).withColumn(
    "tasa_averias_pct", spark_round((col("averias_menos_1min") / col("total_salidas")) * 100, 2)
).filter(col("total_salidas") >= 100).orderBy(col("averias_menos_1min").desc())

df_gold_anomalies.write.mode("overwrite").parquet("/opt/dimencion/gold_station_anomalies")
print(" 3. Guardado: /opt/dimencion/gold_station_anomalies")

# 4. Data Mart: Series Temporales y Media Móvil de 7 días
window_7d = Window.orderBy("start_date").rowsBetween(-6, 0)
df_daily = df_silver.groupBy("start_date").agg(
    spark_round(avg("duration_minutes"), 2).alias("duracion_promedio_dia"),
    count("ride_id").alias("total_viajes_dia")
)

df_gold_timeseries = df_daily.withColumn(
    "media_movil_7d_duracion", spark_round(avg("duracion_promedio_dia").over(window_7d), 2)
)

df_gold_timeseries.write.mode("overwrite").parquet("/opt/dimencion/gold_daily_timeseries")
print(" 4. Guardado: /opt/dimencion/gold_daily_timeseries")

In [ ]:
print("=== DATA MART: RIESGOS TARIFARIOS (MUESTRA) ===")
spark.read.parquet("/opt/dimencion/gold_tariff_risks").show(8, truncate=False)

if hasattr(best_model, "coefficients"):
    print("=== PARÁMETROS DEL MODELO LINEAL PARA STREAMLIT ===")
    print(f"Intercept: {float(best_model.intercept):.4f}")
    for i, c in enumerate(best_model.coefficients):
        print(f"Coeficiente Feature {i:02d}: {float(c):.6f}")

## Síntesis Metodológica para la Sustentación

1. **Garantía Metodológica:** Ausencia total de *data leakage* al ajustar codificadores vectoriales exclusivamente sobre el conjunto de entrenamiento.
2. **Aporte Real del Modelo:** Superación cuantificada frente al modelo nulo (*baseline* de media fija), sustentada con reducción de RMSE/MAE y monitoreo de MAPE.
3. **Distribución Asimétrica:** Uso de percentiles (p50, p75) y rango intercuartílico (IQR) para blindar la toma de decisiones frente al sesgo inducido por viajes atípicos.
4. **Accionabilidad Operativa:** Segmentación en Data Marts independientes que responden directamente a la reasignación de flota, auditoría de sobrecostos por excedente de tiempo y alerta temprana de bicicletas dañadas.